In [9]:
import os

import numpy as np
import pandas as pd 

from azureml.core import Workspace, Dataset, Model, Environment

from azureml.core.model import InferenceConfig
from azureml.core.webservice import AksWebservice

from azureml.opendatasets import NoaaIsdWeather

from datetime import datetime, timedelta

In [2]:
ws = Workspace.from_config()

In [3]:
dset = Dataset.get_by_name(ws, 'my-dataset')

In [5]:
model = Model.register(workspace=ws, 
                       model_path='model.pkl',
                       model_name='my-model',
                       tags={'example': 'text'},
                       description='Example description'
                      )

Registering model my-model


In [8]:
inference_config = InferenceConfig(runtime='python',
                                   entry_script='scoring_file_v_1_0_0.py',
                                   conda_file='conda_env_v_1_0_0.yml')

In [10]:
from azureml.core.compute import AksCompute, ComputeTarget

# Use the default configuration (you can also provide parameters to customize this).
# For example, to create a dev/test cluster, use:
# prov_config = AksCompute.provisioning_configuration(cluster_purpose = AksCompute.ClusterPurpose.DEV_TEST)
prov_config = AksCompute.provisioning_configuration()

aks_name = 'my-aks'
# Create the cluster
aks_target = ComputeTarget.create(workspace = ws,
                                    name = aks_name,
                                    provisioning_configuration = prov_config)

# Wait for the create process to complete
aks_target.wait_for_completion(show_output = True)

Creating......................................................................................................................................................................................
SucceededProvisioning operation finished, operation "Succeeded"


In [16]:
from azureml.core.webservice import AksWebservice, Webservice
from azureml.core.model import Model

aks_target = AksCompute(ws,"my-aks")
# If deploying to a cluster configured for dev/test, ensure that it was created with enough
# cores and memory to handle this deployment configuration. Note that memory is also used by
# things such as dependencies and AML components.
deployment_config = AksWebservice.deploy_configuration(cpu_cores = 1, memory_gb = 1)
service = Model.deploy(ws, "my-service", [model], inference_config, deployment_config, aks_target)
service.wait_for_deployment(show_output = True)
print(service.state)
print(service.get_logs())

ERROR - Error, there is already a service with name my-service found in workspace demoWS



WebserviceException: WebserviceException:
	Message: Error, there is already a service with name my-service found in workspace demoWS
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Error, there is already a service with name my-service found in workspace demoWS"
    }
}

In [1]:
service.delete()

NameError: name 'service' is not defined

In [15]:
service.get_logs()

'/bin/bash: /azureml-envs/azureml_98a094438085301196d78c6383452119/lib/libtinfo.so.5: no version information available (required by /bin/bash)\n/bin/bash: /azureml-envs/azureml_98a094438085301196d78c6383452119/lib/libtinfo.so.5: no version information available (required by /bin/bash)\n/bin/bash: /azureml-envs/azureml_98a094438085301196d78c6383452119/lib/libtinfo.so.5: no version information available (required by /bin/bash)\n2019-09-27T23:42:32,952778434+00:00 - rsyslog/run \n/bin/bash: /azureml-envs/azureml_98a094438085301196d78c6383452119/lib/libtinfo.so.5: no version information available (required by /bin/bash)\nbash: /azureml-envs/azureml_98a094438085301196d78c6383452119/lib/libtinfo.so.5: no version information available (required by bash)\n2019-09-27T23:42:32,956563694+00:00 - nginx/run \n2019-09-27T23:42:32,956654598+00:00 - gunicorn/run \n2019-09-27T23:42:32,957332026+00:00 - iot-server/run \n/usr/sbin/nginx: /azureml-envs/azureml_98a094438085301196d78c6383452119/lib/libcrypt